In [1]:
import xarray as xr
from glob import glob
from braceexpand import braceexpand
from dask.distributed import Client
from dask_jobqueue import SLURMCluster

In [2]:
def braced_glob(path):
    l = []
    for x in braceexpand(path):
        l.extend(glob(x))          
    return l

In [3]:
min_jobs = 4
max_jobs = 10
memory = '64GB'
walltime = '06:00:00'

cluster = SLURMCluster(
    account='rrg-gachon',
    cores=1,
    memory=memory,
    walltime=walltime,
    job_script_prologue=[
        "module load python/3.11 mpi4py/4.0.3 gcc arrow/14.0.1 scipy-stack/2024a ipykernel/2024a geos proj",
        "source /home/vdemeyer/py3/bin/activate"
    ],
    log_directory='/home/vdemeyer/DATA_COMPUTING/JOBS/DASK',
    scheduler_options={"dashboard_address": "localhost:8787"} # http://localhost:8787 in dask extension
)
cluster.adapt(minimum=min_jobs, maximum=max_jobs)
client = Client(cluster)
client.wait_for_workers(min_jobs)
client

<Client: 'tcp://10.80.49.2:46727' processes=0 threads=0, memory=0 B>

In [4]:
variable = 'Precipitation' # 'Wind'

In [ ]:
base_path = '/home/vdemeyer/projects/rrg-gachon/vdemeyer/ERA5'

paths = {
    'Wind': {
        'input': f'{base_path}/WIND/Magnitude/*/*/era5_wind10_ll_*_1h.nc4',
        'output': f'{base_path}/WIND/Magnitude/era5_wind10_CORDEX_NA_1979-2023.zarr'
    },
    'Precipitation': {
        'input': f'{base_path}/PR/*/*/era5_tp_ll_*_1h.nc4',
        'output': f'{base_path}/PR/era5_tp_CORDEX_NA_1979-2023.zarr'
    }
}

if variable in paths:
    filenames = sorted(braced_glob(paths[variable]['input']))
    output_file = paths[variable]['output']

'/home/vdemeyer/projects/rrg-gachon/vdemeyer/ERA5/PR/era5_tp_CORDEX_NA_1979-2023.zarr'

In [ ]:
ds = xr.open_mfdataset(filenames, combine='by_coords')
ds = ds.assign_coords({'longitude': (((ds['longitude'] + 180) % 360) - 180)})
ds = ds.sortby(ds['longitude'])
ds = ds.sel(longitude=slice(-171, -23), latitude=slice(76, 12))
ds = ds.chunk({'time': 900, 'longitude': 593, 'latitude': 257})
ds

<xarray.Dataset> Size: 477GB
Dimensions:    (time: 391536, latitude: 257, longitude: 593)
Coordinates:
  * latitude   (latitude) float32 1kB 76.0 75.75 75.5 75.25 ... 12.5 12.25 12.0
  * time       (time) datetime64[ns] 3MB 1979-01-01 ... 2023-08-31T23:00:00
  * longitude  (longitude) float32 2kB -171.0 -170.8 -170.5 ... -23.25 -23.0
Data variables:
    tp         (time, latitude, longitude) float64 477GB dask.array<chunksize=(1, 257, 593), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6
    history:      2023-07-29 06:32:25 GMT by grib_to_netcdf-2.25.1: /opt/ecmw...

In [13]:
if variable == 'Precipitation':
    ds['tp'] = ds.tp * 1000.

In [14]:
ds.to_zarr(output_file, mode="w", consolidated=True)

print(f"NA CORDEX Dataset file created at {output_file}")

/home/vdemeyer/py3/lib/python3.11/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 458.57 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


NA CORDEX Dataset file created at /home/vdemeyer/projects/rrg-gachon/vdemeyer/ERA5/PR/era5_tp_CORDEX_NA_1979-2023.zarr


In [15]:
cluster.close()
client.close()